# Problem Map 3.0 Troubleshooting Atlas  
## Demo 1 · F1 Grounding Anchor Recovery

### What this experiment is
This notebook is an MVP **route-first repair** experiment.

It is designed to show that:

> not every wrong answer is the same kind of failure  
> some failures break at **Grounding & Evidence Integrity** first  
> so the first repair move should target the **evidence anchor**, not generic prompt tweaking or generic reasoning pressure

---

## What you should expect to see

### Replay mode
You do **not** need an API key.

You will see:

- how the baseline fails
- why Atlas routes the case to **F1**, not **F5**
- what the first repair move is
- how the answer shifts from a wrong-anchor state toward the correct answer

### Live mode
You **do** need an API key.

You will see:

- a baseline prompt that is vulnerable to the wrong anchor
- a repaired prompt that uses the correct anchor
- a before / after contrast

---

## Why an API key is needed
Only **Live mode** needs it.

That is because:

- replay mode only uses fixed fixtures
- live mode actually calls the model twice
  - once for the baseline
  - once for the repaired version

If you only want to understand the demo, replay mode is enough.

---

## What counts as success
The strongest version of this demo is:

- baseline is more likely to drift toward **Lite**
- repaired is more likely to answer **Pro**

But a softer result is still acceptable:

- baseline abstains or becomes uncertain
- repaired still moves clearly to **Pro**

That still proves the key point:

> anchor correction changes the output path

---

## Why this notebook uses GPT-4o mini
This version uses **gpt-4o-mini** because it is cheaper and fast enough for MVP reproduction.

The goal here is not to build the most expensive benchmark.
The goal is to create a small, repeatable, understandable demo.


In [ ]:
%pip -q install openai

In [ ]:
# Problem Map 3.0 Troubleshooting Atlas
# Demo 1 · F1 Grounding Anchor Recovery

import os
from getpass import getpass
from pprint import pprint

try:
    from openai import OpenAI
except Exception:
    OpenAI = None


In [ ]:
MODE = "live"   # choose: "replay" or "live"
MODEL = "gpt-4o-mini"

print(f"MODE = {MODE}")
print(f"MODEL = {MODEL}")


In [ ]:
INPUT_CASE = {
    "demo_id": "demo_f1_grounding_anchor",
    "demo_version": "v1",
    "case_id": "f1_anchor_case_001",
    "title": "Fluent answer attached to the wrong evidence anchor",
    "task_type": "retrieval_grounded_qa",
    "family_target": {
        "primary_family": "F1",
        "secondary_family": "F5",
        "best_current_fit": "F1_N01 Retrieval Anchor Drift",
        "broken_invariant": "evidence_anchor_integrity_broken"
    },
    "user_question": "According to the official release notes, which product tier includes Semantic Refraction and Tension Field?",
    "gold_answer": "Pro",
    "chunks": [
        {
            "chunk_id": "chunk_001",
            "label": "Lite tier overview",
            "relevance_role": "distractor_semantic_neighbor",
            "text": "Lite includes Semantic Gravity Well, Quick Blah, Semantic Tree Memory, and TXT-Blah Blah Blah Lite with 50 answers. It is intended for beginners."
        },
        {
            "chunk_id": "chunk_002",
            "label": "Pro tier overview",
            "relevance_role": "correct_anchor",
            "text": "Pro includes all Lite features plus Semantic Refraction, Tension Field, and Orbital Drift of Meaning."
        },
        {
            "chunk_id": "chunk_003",
            "label": "General product summary",
            "relevance_role": "broad_context",
            "text": "The product line includes multiple tiers for different user groups, from beginner-friendly entry versions to more structured reasoning editions."
        }
    ]
}

REPLAY_OUTPUTS = {
    "baseline_answer": "Lite includes those features.",
    "baseline_anchor": "chunk_001",
    "repaired_answer": "Pro includes all Lite features plus Semantic Refraction, Tension Field, and Orbital Drift of Meaning.",
    "repaired_short_answer": "Pro",
    "repaired_anchor": "chunk_002"
}

EXPECTED_OUTPUT = {
    "primary_family": "F1",
    "secondary_family": "F5",
    "best_current_fit": "F1_N01 Retrieval Anchor Drift",
    "broken_invariant": "evidence_anchor_integrity_broken",
    "short_answer": "Pro"
}


In [ ]:
def print_section(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def print_chunks(chunks):
    for item in chunks:
        print(f"[{item['chunk_id']}] {item['label']} | role={item['relevance_role']}")
        print(item["text"])
        print("-" * 80)

def safe_output_text(resp):
    if resp is None:
        return None
    text = getattr(resp, "output_text", None)
    if text:
        return text
    try:
        return str(resp.output[0].content[0].text)
    except Exception:
        return str(resp)


In [ ]:
print_section("1. Case overview")
print("Question:")
print(INPUT_CASE["user_question"])

print_section("2. Retrieved chunks")
print_chunks(INPUT_CASE["chunks"])

print_section("3. Atlas routing target")
pprint(INPUT_CASE["family_target"])


In [ ]:
print_section("Replay mode · baseline")
print("Baseline answer:", REPLAY_OUTPUTS["baseline_answer"])
print("Baseline anchor:", REPLAY_OUTPUTS["baseline_anchor"])

print_section("Replay mode · repaired")
print("Repaired answer:", REPLAY_OUTPUTS["repaired_answer"])
print("Short answer:", REPLAY_OUTPUTS["repaired_short_answer"])
print("Repaired anchor:", REPLAY_OUTPUTS["repaired_anchor"])


In [ ]:
client = None

if MODE == "live":
    if OpenAI is None:
        raise RuntimeError("The openai package is not available. Please rerun the install cell first.")
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    client = OpenAI()
    print("OpenAI client ready.")
else:
    print("Replay mode selected. No API key required.")


In [ ]:
baseline_prompt = f'''
You are answering a product question from incomplete retrieved notes.

Question:
{INPUT_CASE["user_question"]}

Retrieved notes:
1. {INPUT_CASE["chunks"][0]["text"]}
2. {INPUT_CASE["chunks"][2]["text"]}

Choose the single most likely product tier from the retrieved notes.
Answer with one tier label only.
Do not say "unknown", "not specified", or "insufficient information".
'''.strip()

repaired_prompt = f'''
You are answering a product question from verified evidence.

Question:
{INPUT_CASE["user_question"]}

Verified evidence:
1. {INPUT_CASE["chunks"][1]["text"]}

Answer with one tier label only.
'''.strip()

print_section("Baseline prompt")
print(baseline_prompt)

print_section("Repaired prompt")
print(repaired_prompt)


In [ ]:
def run_text_call(prompt: str, model: str):
    response = client.responses.create(
        model=model,
        input=prompt
    )
    return safe_output_text(response)

if MODE == "live":
    print_section("Live mode · baseline model output")
    live_baseline = run_text_call(baseline_prompt, MODEL)
    print(live_baseline)

    print_section("Live mode · repaired model output")
    live_repaired = run_text_call(repaired_prompt, MODEL)
    print(live_repaired)
else:
    print("Set MODE = 'live' above if you want to run real model calls.")


In [ ]:
print_section("Expected effect checklist")
print("You should verify the following:")
print("1. Baseline is more vulnerable to answering Lite or another wrong-anchor answer.")
print("2. Repaired version is more likely to answer Pro.")
print("3. The important shift is not prompt length.")
print("4. The important shift is evidence anchor correction.")
print("5. This is why Atlas routes the case to F1 before F5.")

print_section("Expected success contract")
pprint(EXPECTED_OUTPUT)


---

## Back to the main page

Read the full product page here:  
[Problem Map 3.0 Troubleshooting Atlas](https://github.com/onestardao/WFGY/blob/main/ProblemMap/wfgy-ai-problem-map-troubleshooting-atlas.md)

If you like the project, **star the repo** ⭐
